# 7CS353 Machine Learning Lab — Assignment 8
### Decision Tree and Random Forest Classification

Name: Atharv V Kulkarni
PRN: 245109074
Class: TY CSE(B)

## B1. Load and explore

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
np.random.seed(42)

In [ ]:
iris = load_iris()
x = iris.data
y = iris.target
feature_names = iris.feature_names
class_names = iris.target_names
df = pd.DataFrame(x, columns=feature_names)
df["species"] = pd.Categorical.from_codes(y, class_names)
print(df.shape)
print(df["species"].value_counts())
print(df.describe())

In [ ]:
sns.pairplot(df, hue="species", corner=True)
plt.show()

**Solution:**

_(fill in after viewing the pair plot — name the pair of features where the three species clusters look most clearly separated)_

## B2. Impurity from scratch

In [ ]:
def entropy(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p))

def gini(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return 1 - np.sum(p**2)

In [ ]:
petal_length = x[:, 2]
mask = petal_length <= 2.45
left = y[mask]
right = y[~mask]

h_parent = entropy(y)
h_left = entropy(left)
h_right = entropy(right)
info_gain = h_parent - (len(left)/len(y))*h_left - (len(right)/len(y))*h_right

print("entropy full set:", h_parent, "gini full set:", gini(y))
print("entropy left (petal_length<=2.45):", h_left, "gini left:", gini(left))
print("entropy right (petal_length>2.45):", h_right, "gini right:", gini(right))
print("information gain:", info_gain)

**Solution:**

_(fill in after running — confirm whether petal_length <= 2.45 matches the root split in Figure 1, and comment on whether this information gain value indicates a strong split)_

## B3. Decision tree — fit and visualise

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
dt = DecisionTreeClassifier(max_depth=3, criterion="entropy", random_state=42)
dt.fit(x_train, y_train)

In [ ]:
fig, ax = plt.subplots(figsize=(14,8))
plot_tree(dt, feature_names=feature_names, class_names=class_names, filled=True, ax=ax)
ax.set_title("decision tree (max_depth=3, criterion=entropy)")
plt.show()

In [ ]:
print(export_text(dt, feature_names=feature_names))

In [ ]:
y_pred_dt = dt.predict(x_test)
print(classification_report(y_test, y_pred_dt, target_names=class_names))

In [ ]:
cm_dt = confusion_matrix(y_test, y_pred_dt)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("decision tree confusion matrix")
plt.show()

## B4. Depth analysis

In [ ]:
train_acc = []
test_acc = []
depths = range(1, 11)
for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, criterion="entropy", random_state=42)
    model.fit(x_train, y_train)
    train_acc.append(accuracy_score(y_train, model.predict(x_train)))
    test_acc.append(accuracy_score(y_test, model.predict(x_test)))

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(list(depths), train_acc, marker="o", label="train accuracy", color="royalblue")
ax.plot(list(depths), test_acc, marker="o", label="test accuracy", color="crimson")
ax.set_xlabel("max depth")
ax.set_ylabel("accuracy")
ax.set_title("training vs test accuracy across tree depth")
ax.legend()
plt.show()

**Solution:**

_(fill in after viewing the curves — identify the depth at which training accuracy approaches 1.0 while test accuracy plateaus or drops, and justify an optimal depth choice)_

## B5. Random forest — fit and evaluate

In [ ]:
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf.fit(x_train, y_train)
print("oob score:", rf.oob_score_)
y_pred_rf = rf.predict(x_test)
print(classification_report(y_test, y_pred_rf, target_names=class_names))

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values()
fig, ax = plt.subplots(figsize=(7,5))
ax.barh(importances.index, importances.values, color="seagreen")
ax.set_xlabel("importance")
ax.set_title("random forest feature importances")
plt.show()

**Solution:**

_(fill in after running — compare the most important features here with the feature pair that looked most separating in the B1 pair plot)_

## B6. Hyperparameter tuning

In [ ]:
param_grid = {"n_estimators": [50, 100, 200], "max_depth": [None, 5, 10, 15]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
grid.fit(x_train, y_train)
print("best params:", grid.best_params_)
print("best cv accuracy:", grid.best_score_)

In [ ]:
rf_tuned = grid.best_estimator_
y_pred_tuned = rf_tuned.predict(x_test)
print(classification_report(y_test, y_pred_tuned, target_names=class_names))

## B7. Comparison and interpretation

In [ ]:
def metrics_row(y_true, y_pred):
    return (accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred, average="macro"),
            recall_score(y_true, y_pred, average="macro"),
            f1_score(y_true, y_pred, average="macro"))

print("decision tree (depth=3):", metrics_row(y_test, y_pred_dt))
print("random forest (default):", metrics_row(y_test, y_pred_rf))
print("random forest (tuned):", metrics_row(y_test, y_pred_tuned))

| Model | Test Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| Decision Tree (depth=3) | _ | _ | _ | _ |
| Random Forest (default) | _ | _ | _ | _ |
| Random Forest (tuned) | _ | _ | _ | _ |

_(fill in the blanks with the printed values from the cell above)_

**Solution:**

_(write your final interpretation after running — cover (1) which model you would deploy and why, (2) what the feature importances tell you about the Iris features, and (3) a scenario where a single decision tree would be preferable to a random forest despite lower accuracy)_